In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, Lasso
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv("components.csv")
X = df.drop(columns=["measurement","no","id"]).values

y = df["measurement"].values

In [ ]:
# Leave-One-Out Cross-Validation
loo = LeaveOneOut()

def evaluate_model(model):
    preds = []
    truth = []
    stds = []

    for train_idx, test_idx in loo.split(X):
        model.fit(X[train_idx], y[train_idx])

        # GPR returns uncertainty
        if hasattr(model.named_steps["reg"], "predict"):
            reg = model.named_steps["reg"]
        else:
            reg = model.named_steps["gpr"]

        if isinstance(reg, GaussianProcessRegressor):
            mean, std = model.predict(X[test_idx], return_std=True)
            preds.append(mean[0])
            stds.append(std[0])
        else:
            preds.append(model.predict(X[test_idx])[0])
            stds.append(None)

        truth.append(y[test_idx][0])

    rmse = np.sqrt(mean_squared_error(truth, preds))
    return rmse, stds, truth, preds

# --- Models ---
ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("reg", Ridge(alpha=1.0))
])

lasso = Pipeline([
    ("scaler", StandardScaler()),
    ("reg", Lasso(alpha=0.01, max_iter=10000))
])

kernel = ConstantKernel(1.0, (1e-4, 1e4)) * \
         RBF(length_scale=1.0, length_scale_bounds=(1e-8, 1e4)) + \
         WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-6, 1e1))

gpr = Pipeline([
    ("scaler", StandardScaler()),
    ("reg", GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10)) # gpr
])

# --- Evaluate ---
rmse_ridge, _, truth_ridge, preds_ridge = evaluate_model(ridge)
rmse_lasso, _, truth_lasso, preds_lasso = evaluate_model(lasso)
rmse_gpr, stds_gpr, truth_gpr, preds_gpr = evaluate_model(gpr)

print("Ridge RMSE:", rmse_ridge)
print("Lasso RMSE:", rmse_lasso)
print("GPR RMSE:", rmse_gpr)
print("GPR mean predictive std:", np.nanmean(stds_gpr))

# combined figure (vertical arrangement)
fig, axes = plt.subplots(3, 1, figsize=(6, 18))
models = ["Ridge", "Lasso", "GPR"]
truths = [truth_ridge, truth_lasso, truth_gpr]
preds = [preds_ridge, preds_lasso, preds_gpr]
colors = ["blue", "green", "red"]

for ax, m, t, p, c in zip(axes, models, truths, preds, colors):
    ax.scatter(t, p, color=c, alpha=0.7)
    ax.plot([min(t), max(t)], [min(t), max(t)], "r--")
    ax.set_xlabel("Actual Inrush")
    ax.set_ylabel("Predicted Inrush")
    ax.set_title(f"{m} Predictions")
    ax.grid(alpha=0.3)

fig.suptitle("LOOCV Predictions by Model")
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


In [ ]:
ridge_model = ridge.named_steps["reg"]
scaler = ridge.named_steps["scaler"]

coef = ridge_model.coef_
feature_names = df.columns[:-1].drop(["no","id"])

#for name, c in zip(feature_names, coef):
#    print(f"{name:20s} {c:+.4f}")

importance = sorted(zip(feature_names, coef), key=lambda x: abs(x[1]), reverse=True)

for name, c in importance:
    print(f"{name:20s} {c:+.4f}")



In [ ]:
print(ridge.named_steps["reg"].coef_)
sum = 0 
for i in ridge.named_steps["reg"].coef_:
    sum += abs(i)

print(sum)
    

In [ ]:
preds = ridge.predict(df.drop(columns=["measurement","no","id"]).values)    

print("Predicted inrush currents for new data:")
#for i, pred in enumerate(preds):
#    print(f"Sample {i+1}: {pred:.2f}")

# for easy copy-paste
for i, pred in enumerate(preds):
    print(f"{pred:.2f} ")


In [ ]:
#df.drop(columns=["measurement","no","id"]).values

In [ ]:
feature_names

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ridge_model = ridge.named_steps["reg"]
feature_names = df.columns[2:-1]
coef = ridge_model.coef_

plt.figure(figsize=(10, 6))
plt.barh(feature_names, coef)
plt.xlabel("Standardized Coefficient")
plt.title("Ridge Regression Feature Importance")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

X_mean = X.mean(axis=0)
print(X_mean)

fig, axes = plt.subplots(7, 2, figsize=(12, 20))
axes = axes.flatten()

for i, ax in enumerate(axes[:len(feature_names)]):
    # vary only feature i
    x_vals = np.linspace(X[:, i].min(), X[:, i].max(), 50)
    X_temp = np.tile(X_mean, (50, 1))
    X_temp
    X_temp[:, i] = x_vals

    y_pred = ridge.predict(X_temp)

    ax.scatter(X[:, i], y, alpha=0.6, label="Actual")
    ax.plot(x_vals, y_pred, color="red", label="Model")
    ax.set_title(feature_names[i])
    ax.set_xlabel(feature_names[i])
    ax.set_ylabel("Inrush Current")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
X_mean = X.mean(axis=0)

fig, axes = plt.subplots(7, 2, figsize=(12, 20))
axes = axes.flatten()

for i, ax in enumerate(axes[:len(feature_names)]):
    # vary only feature i
    x_vals = np.linspace(X[:, i].min(), X[:, i].max(), 50)
    X_temp = np.tile(X_mean, (50, 1))
    X_temp[:, i] = x_vals

    y_pred = gpr.predict(X_temp)

    ax.scatter(X[:, i], y, alpha=0.6, label="Actual")
    ax.plot(x_vals, y_pred, color="red", label="Model")
    ax.set_title(feature_names[i])
    ax.set_xlabel(feature_names[i])
    ax.set_ylabel("Inrush Current")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,7))

# Ridge
plt.scatter(truth_ridge, preds_ridge, label="Ridge", color="blue", alpha=0.7)

# GPR
plt.scatter(truth_gpr, preds_gpr, label="GPR", color="red", alpha=0.7)

# Perfect prediction line
min_val = min(truth_ridge + truth_gpr)
max_val = max(truth_ridge + truth_gpr)
plt.plot([min_val, max_val], [min_val, max_val], "k--", label="Perfect")

plt.xlabel("Actual Inrush Current")
plt.ylabel("Predicted Inrush Current")
plt.title("Ridge vs GPR — LOOCV Predictions")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:

errors_gpr = np.abs(np.array(preds_gpr) - np.array(truth_gpr))

plt.figure(figsize=(7,5))
plt.scatter(stds_gpr, errors_gpr, color="purple", alpha=0.7)
plt.xlabel("GPR Predictive Std (Uncertainty)")
plt.ylabel("Absolute Prediction Error")
plt.title("Does GPR Know When It’s Wrong?")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
coef = ridge.named_steps["reg"].coef_
feature_names = df.columns[:-1].drop(["no","id"])

plt.figure(figsize=(10,6))
plt.barh(feature_names, coef)
plt.title("Ridge Feature Importance (Standardized Coefficients)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()